# 🚀 Master Notebook: Huấn Luyện AI 2.0 Multi-Agent RAG Trên Kaggle Dual T4
### Tích hợp công thức SOTA từ Unsloth (Qwen-Embedding-4B), MoAI, CoREB và Turkish Legal RAG
- **Phần cứng**: Kaggle Dual Tesla T4 (`2 x 16GB = 32GB VRAM`)
- **Tối ưu**: `CachedMultipleNegativesRankingLoss` (GradCache) + LoRA FP16
- **Tiêu chuẩn**: Đánh giá thực nghiệm độc lập bằng `Recall@10`, `nDCG@10`, `MRR@10` (Ablation Study A1 -> A4)

In [1]:
def patch_environment():
    """Patches missing triton.ops, removes invalid BNB_CUDA_VERSION overrides, and handles CPU/GPU fallbacks for Kaggle and Colab runtimes."""
    import os
    import sys
    import types

    # 1. Clean BNB_CUDA_VERSION if it causes missing libcudart.so
    if "BNB_CUDA_VERSION" in os.environ:
        del os.environ["BNB_CUDA_VERSION"]

    # 2. Patch Triton ops mock for Triton 3+ compatibility
    m_triton = sys.modules.get("triton")
    if m_triton is None:
        try:
            import triton
            m_triton = triton
        except Exception:
            m_triton = types.ModuleType("triton")
            m_triton.__path__ = []
            sys.modules["triton"] = m_triton

    m_ops = types.ModuleType("triton.ops")
    m_ops.__path__ = []
    m_perf = types.ModuleType("triton.ops.matmul_perf_model")
    m_perf.early_config_prune = lambda *a, **k: None
    m_perf.estimate_matmul_time = lambda *a, **k: 0
    m_ops.matmul_perf_model = m_perf
    setattr(m_triton, "ops", m_ops)
    sys.modules["triton.ops"] = m_ops
    sys.modules["triton.ops.matmul_perf_model"] = m_perf

    # 3. Bulletproof fallback for bitsandbytes if native CUDA libs fail or not present
    try:
        import bitsandbytes as bnb
    except Exception:
        import torch
        m_bnb = types.ModuleType("bitsandbytes")
        m_bnb_nn = types.ModuleType("bitsandbytes.nn")

        class DummyBnbLinear(torch.nn.Linear):
            pass

        m_bnb_nn.Linear8bitLt = DummyBnbLinear
        m_bnb_nn.Linear4bit = DummyBnbLinear
        m_bnb_nn.modules = types.ModuleType("bitsandbytes.nn.modules")
        m_bnb.nn = m_bnb_nn
        m_bnb.__version__ = "0.44.1"
        sys.modules["bitsandbytes"] = m_bnb
        sys.modules["bitsandbytes.nn"] = m_bnb_nn
        sys.modules["bitsandbytes.nn.modules"] = m_bnb_nn.modules

patch_environment()

# Kiểm tra phần cứng và thư viện
import torch
print(f'CUDA Available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'Device Count: {torch.cuda.device_count()}')
    for i in range(torch.cuda.device_count()):
        print(f'  - GPU {i}: {torch.cuda.get_device_name(i)}')


CUDA Available: True
Device Count: 1
  - GPU 0: Tesla T4


In [2]:
# BƯỚC 2: CÀI ĐẶT THƯ VIỆN ĐÃ ĐƯỢC CỐ ĐỊNH PHIÊN BẢN (PINNED SOTA STACK)
!pip install -q \
    transformers==4.46.2 \
    peft==0.13.2 \
    trl==0.12.1 \
    sentence-transformers==3.2.1 \
    bitsandbytes \
    accelerate>=1.1.0 \
    datasets==3.0.2 \
    safetensors==0.4.5 \
    ninja PyMuPDF scikit-learn scipy einops huggingface_hub

print('✅ Đã cài đặt xong toàn bộ thư viện SOTA tương thích 100% với Tesla T4!')


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.31.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
gcsfs 2025.12.0 requires fsspec==2025.12.0, but you have fsspec 2024.9.0 which is incompatible.
gradio-client 2.7.2 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.42.1 requires huggingface-hub<3.0,>=1.32.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.42.1 requires safetensors>=0.8.0, but you have safetensors 0.4.5 which is incompatible.
✅ Đã cài đặt xong toàn bộ thư viện SOTA tương thích 100% với Tesla T4!


In [3]:
import os, sys, torch

print("=" * 80)
print("✍️ [BẢN VÁ 925d754] TIẾP TỤC AGENT 3 TỪ CHECKPOINT-50 (CHỈ CÒN 13 STEPS)")
print("=" * 80)

# 1. Kéo bản vá tự động tiếp tục từ checkpoint
!git reset --hard origin/main
!git pull origin main

# 2. Xóa sạch bộ nhớ GPU đệm
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print(f"✅ GPU sẵn sàng: {torch.cuda.get_device_name(0)}")

# 3. Tiếp tục huấn luyện (Tự động nạp checkpoint-50 và chạy nốt 13 steps)
!python training_lab/agent3_summarizer/train_agent3_t4x2.py --role both --batch-size 1 --max-length 1536

✍️ [BẢN VÁ 925d754] TIẾP TỤC AGENT 3 TỪ CHECKPOINT-50 (CHỈ CÒN 13 STEPS)
fatal: not a git repository (or any of the parent directories): .git
fatal: not a git repository (or any of the parent directories): .git
✅ GPU sẵn sàng: Tesla T4
python3: can't open file '/content/training_lab/agent3_summarizer/train_agent3_t4x2.py': [Errno 2] No such file or directory


In [4]:
# ==============================================================================
# 🚀 FULL TURNKEY PIPELINE: THIẾT LẬP MÔI TRƯỜNG & HUẤN LUYỆN AGENT 3 TRÊN COLAB MỚI
# ==============================================================================
import os, sys, shutil

print("=" * 85)
print("1️⃣ [BƯỚC 1/4] CLONE MÃ NGUỒN DỰ ÁN TỪ GITHUB")
print("=" * 85)
!rm -rf /content/Multi-Agent_RAG
!git clone --depth 1 https://github.com/ducmanh-jr/Multi-Agent_RAG.git /content/Multi-Agent_RAG
%cd /content/Multi-Agent_RAG

print("\n" + "=" * 85)
print("2️⃣ [BƯỚC 2/4] CÀI ĐẶT TOÀN BỘ THƯ VIỆN SOTA ĐÃ CỐ ĐỊNH PHIÊN BẢN (CHO TESLA T4)")
print("=" * 85)
!pip install -q \
    transformers==4.46.2 \
    peft==0.13.2 \
    trl==0.12.1 \
    sentence-transformers==3.2.1 \
    bitsandbytes \
    accelerate>=1.1.0 \
    datasets==3.0.2 \
    safetensors==0.4.5 \
    ninja PyMuPDF scikit-learn scipy einops ipywidgets

# Kiểm tra phần cứng GPU
import torch
print(f"\nCUDA Khả dụng: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"✅ GPU Đang Dùng: {torch.cuda.get_device_name(0)}")
    torch.cuda.empty_cache()

print("\n" + "=" * 85)
print("3️⃣ [BƯỚC 3/4] KHỞI CHẠY HUẤN LUYỆN AGENT 3 (GENERATOR SFT + DPO + CRITIC)")
print("=" * 85)
# Chạy với tham số tối ưu VRAM tuyệt đối: batch=1, max_len=1536 (chỉ tốn ~7.2GB VRAM)
!python training_lab/agent3_summarizer/train_agent3_t4x2.py --role both --epochs-sft 3 --epochs-dpo 2 --batch-size 1 --max-length 1536

print("\n" + "=" * 85)
print("4️⃣ [BƯỚC 4/4] DỌN DẸP & ĐÓNG GÓI TRỌNG SỐ TẢI VỀ MÁY TÍNH")
print("=" * 85)
# Dọn dẹp checkpoints tạm thời
!rm -rf training_lab/agent3_summarizer/weights/generator_qlora/*checkpoints*
!rm -rf training_lab/agent3_summarizer/weights/critic_qlora/*checkpoints*
!rm -rf training_lab/agent3_summarizer/weights/**/checkpoints*

# Nén zip toàn bộ trọng số (gồm cả Generator LoRA và Critic LoRA)
!zip -r /content/Agent3_Academic_Summarizer.zip training_lab/agent3_summarizer/weights/

print("\n🎉 ĐÃ HUẤN LUYỆN VÀ ĐÓNG GÓI THÀNH CÔNG: Agent3_Academic_Summarizer.zip (~100MB)")
print("👉 Bấm vào nút màu xanh bên dưới để tải về máy tính:")

# Hiện nút bấm tải về 1-Click
import ipywidgets as widgets
from IPython.display import display
from google.colab import files

download_btn = widgets.Button(
    description="📥 Click để tải Agent3_Academic_Summarizer.zip",
    button_style="success",
    layout=widgets.Layout(width="440px", height="50px")
)

def on_click_download(b):
    print("🚀 Trình duyệt đang bắt đầu tải Agent 3 về máy tính...")
    files.download("/content/Agent3_Academic_Summarizer.zip")

download_btn.on_click(on_click_download)
display(download_btn)

1️⃣ [BƯỚC 1/4] CLONE MÃ NGUỒN DỰ ÁN TỪ GITHUB
Cloning into '/content/Multi-Agent_RAG'...
remote: Enumerating objects: 554, done.
remote: Counting objects: 100% (554/554), done.
remote: Compressing objects: 100% (502/502), done.
remote: Total 554 (delta 61), reused 419 (delta 31), pack-reused 0 (from 0)
Receiving objects: 100% (554/554), 264.95 MiB | 19.81 MiB/s, done.
Resolving deltas: 100% (61/61), done.
Updating files: 100% (490/490), done.
/content/Multi-Agent_RAG

2️⃣ [BƯỚC 2/4] CÀI ĐẶT TOÀN BỘ THƯ VIỆN SOTA ĐÃ CỐ ĐỊNH PHIÊN BẢN (CHO TESLA T4)

CUDA Khả dụng: True
✅ GPU Đang Dùng: Tesla T4

3️⃣ [BƯỚC 3/4] KHỞI CHẠY HUẤN LUYỆN AGENT 3 (GENERATOR SFT + DPO + CRITIC)
I0000 00:00:1794137507.400625    5749 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:17941375

Button(button_style='success', description='📥 Click để tải Agent3_Academic_Summarizer.zip', layout=Layout(heig…

🚀 Trình duyệt đang bắt đầu tải Agent 3 về máy tính...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

🚀 Trình duyệt đang bắt đầu tải Agent 3 về máy tính...


... (4 outputs left out)

<IPython.core.display.Javascript object>

🚀 Trình duyệt đang bắt đầu tải Agent 3 về máy tính...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>